# Global Stock Market Development — Colab-Compatible Reproduction

This notebook is a **compatibility-fixed reproduction** of the public `research_code.ipynb` released with:

**Stawarz, M. (2025). _Analysis of global stock market development—Integration of clustering, classification, and Shapley values._ PLOS ONE 20(6): e0326809.**

Original data/code repository: https://doi.org/10.18150/OELMLK

### What was fixed
The released notebook and released CSV do not run together unchanged. This version keeps the paper's core workflow but fixes:
- released CSV/header mismatches;
- malformed `sty` values in the deposited CSV by treating them as missing and imputing them;
- unstable Mahalanobis matrix inversion;
- unresolved Random Forest `PLACEHOLDER_...` values;
- current SHAP multi-class output handling;
- the removed GeoPandas `naturalearth_lowres` helper by using Plotly for the country map.

**Upload `stock_exchanges_data.csv` into the same Colab session before using Runtime → Run all.**


In [ ]:
# Core imports
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import chi2
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
import shap

print("Imports successful.")


In [ ]:
# Load and clean the deposited CSV
DATA_FILE = "stock_exchanges_data.csv"

if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(
        f"{DATA_FILE} was not found. In Colab, click the Files icon on the left "
        "and upload stock_exchanges_data.csv, then run all cells again."
    )

# Read as text first because the deposited file uses semicolons, decimal commas,
# and also contains a few Excel-style 'sty' strings.
raw = pd.read_csv(DATA_FILE, sep=";", encoding="cp1250", dtype=str)
raw.columns = raw.columns.str.strip()

id_cols = ["ExchangeName", "Country"]
numeric_cols = [c for c in raw.columns if c not in id_cols]

df = raw.copy()

for col in numeric_cols:
    s = (
        df[col]
        .astype(str)
        .str.strip()
        .str.replace(",", ".", regex=False)
    )
    # The deposited CSV contains Polish Excel-style date fragments such as
    # 'sty.00' / '22.sty'. They are not valid numeric observations.
    s = s.mask(s.str.contains("sty", case=False, na=False))
    df[col] = pd.to_numeric(s, errors="coerce")

# Reconstruct ratio fields when their components are present.
def fill_ratio(target, numerator, denominator):
    with np.errstate(divide="ignore", invalid="ignore"):
        calculated = df[numerator] / df[denominator]
    df[target] = df[target].fillna(calculated.replace([np.inf, -np.inf], np.nan))

fill_ratio("Capitalization/GDP", "Capitalization", "GDP")
fill_ratio("Value traded (Total)/GDP", "Value traded (EOB Total)", "GDP")
fill_ratio("Share turnover velocity", "Value traded (EOB Total)", "Capitalization")
fill_ratio(
    "Capitalization/Number of listed companies (Total)",
    "Capitalization",
    "Number of listed companies (Total)"
)
fill_ratio(
    "Number of listed companies (Foreign)/Number of listed companies (Total)",
    "Number of listed companies (Foreign)",
    "Number of listed companies (Total)"
)
fill_ratio(
    "Number of listed companies (Domestic)/Population",
    "Number of listed companies (Domestic)",
    "Population"
)

print(f"Loaded {len(df)} stock exchanges and {len(df.columns)} columns.")
print(df[["ExchangeName", "Country"]].head())


In [ ]:
# Paper's 13 candidate indicators + preprocessing
candidate_vars = [
    "Capitalization",
    "Capitalization/GDP",
    "Value traded (EOB Total)",
    "Value traded (Total)/GDP",
    "Share turnover velocity",
    "Capitalization/Number of listed companies (Total)",
    "Number of listed companies (Total)",
    "Number of listed companies (Foreign)/Number of listed companies (Total)",
    "Number of listed companies (Domestic)/Population",
    "Number of listed companies (Domestic)",
    "Number of listed companies (Foreign)",
    "Number of new listings through IPO (Total)",
    "Number of trades (EOB)"
]

missing_before = df[candidate_vars].isna().sum()

# The paper describes imputing sporadic gaps. Median imputation is used here
# so the deposited CSV can be executed reproducibly.
imputer = SimpleImputer(strategy="median")
data_to_model = pd.DataFrame(
    imputer.fit_transform(df[candidate_vars]),
    columns=candidate_vars,
    index=df.index
)

print("Missing values before imputation:")
print(missing_before[missing_before > 0])
print("\nMissing values after imputation:", int(data_to_model.isna().sum().sum()))

# Standardize all candidate indicators.
candidate_scaler = StandardScaler()
candidate_scaled = candidate_scaler.fit_transform(data_to_model)

# Mahalanobis outlier screening at alpha = 0.001.
# pinv is numerically stable when the covariance matrix is near-singular.
cov_matrix = np.cov(candidate_scaled, rowvar=False)
inv_cov_matrix = np.linalg.pinv(cov_matrix)
md2 = np.einsum("ij,jk,ik->i", candidate_scaled, inv_cov_matrix, candidate_scaled)

alpha = 0.001
cutoff = chi2.ppf(1 - alpha, df=len(candidate_vars))
outlier_flag = md2 > cutoff

print(f"Mahalanobis cutoff: {cutoff:.3f}")
print(f"Potential outliers flagged: {int(outlier_flag.sum())} of {len(outlier_flag)}")

# The article describes these as potential outliers that were examined.
# No exclusion list is supplied, so all 82 exchanges are retained for the
# clustering/classification reproduction.


In [ ]:
# Seven indicators selected in the published paper
best_vars = [
    "Capitalization",
    "Capitalization/GDP",
    "Value traded (EOB Total)",
    "Value traded (Total)/GDP",
    "Share turnover velocity",
    "Capitalization/Number of listed companies (Total)",
    "Number of trades (EOB)"
]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(data_to_model[best_vars])
X = pd.DataFrame(X_scaled, columns=best_vars, index=df.index)

# Silhouette and elbow diagnostics for k = 2..10
clusters_range = range(2, 11)
silhouette_scores = []
inertias = []

for k in clusters_range:
    km = KMeans(
        n_clusters=k,
        init="random",
        n_init=100,
        max_iter=100,
        random_state=42
    )
    labels = km.fit_predict(X)
    silhouette_scores.append(silhouette_score(X, labels))
    inertias.append(km.inertia_)
    print(
        f"k={k:2d} | silhouette={silhouette_scores[-1]:.4f} "
        f"| inertia={inertias[-1]:.2f}"
    )

diagnostics = pd.DataFrame({
    "k": list(clusters_range),
    "silhouette": silhouette_scores,
    "inertia": inertias
})

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(diagnostics["k"], diagnostics["silhouette"], marker="o")
ax.axvline(5, linestyle="--")
ax.set_xlabel("Number of clusters")
ax.set_ylabel("Silhouette coefficient")
ax.set_title("Figure 1 reproduction: Silhouette analysis")
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(diagnostics["k"], diagnostics["inertia"], marker="o")
ax.axvline(5, linestyle="--")
ax.set_xlabel("Number of clusters")
ax.set_ylabel("Inertia (distortion)")
ax.set_title("Figure 2 reproduction: Elbow method")
plt.show()

# The paper selects k=5 using the combined silhouette/elbow interpretation.
OPT_K = 5
final_km = KMeans(
    n_clusters=OPT_K,
    init="random",
    n_init=100,
    max_iter=100,
    random_state=42
)
cluster_labels = final_km.fit_predict(X)

results = df[["ExchangeName", "Country"]].copy()
results["cluster"] = cluster_labels
results["cluster_display"] = results["cluster"] + 1
results["silhouette"] = silhouette_samples(X, cluster_labels)

print("\nCluster sizes:")
print(results["cluster_display"].value_counts().sort_index())
print("\nMean silhouette for k=5:", round(float(results["silhouette"].mean()), 4))
print(results.head())


In [ ]:
# Random Forest classifier + SHAP
X_train = X.copy()
y_train = cluster_labels

rf = RandomForestClassifier(random_state=42)

# The deposited notebook contains unresolved PLACEHOLDER values.
# This concrete, compact grid makes the released workflow executable.
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

cv_rf = GridSearchCV(
    rf,
    param_grid,
    cv=2,
    n_jobs=-1
)
cv_rf.fit(X_train, y_train)
best_rf = cv_rf.best_estimator_

print("Best Random Forest parameters:")
print(cv_rf.best_params_)
print("Training accuracy:", round(best_rf.score(X_train, y_train), 4))

explainer = shap.TreeExplainer(best_rf)
shap_values = explainer.shap_values(X_train)

# Normalize current/older SHAP multi-class formats to:
# (n_samples, n_features, n_classes)
if isinstance(shap_values, list):
    shap_array = np.stack(shap_values, axis=-1)
else:
    shap_array = np.asarray(shap_values)

if shap_array.ndim != 3:
    raise RuntimeError(f"Unexpected SHAP array shape: {shap_array.shape}")

print("SHAP array shape:", shap_array.shape)

# Overall feature importance
mean_abs_shap = np.abs(shap_array).mean(axis=(0, 2))
importance = pd.Series(mean_abs_shap, index=best_vars).sort_values()

fig, ax = plt.subplots(figsize=(9, 5))
importance.plot(kind="barh", ax=ax)
ax.set_xlabel("Mean absolute SHAP value")
ax.set_title("Overall Random Forest SHAP feature importance")
plt.tight_layout()
plt.show()


In [ ]:
# Figures 3–7: SHAP interpretation for the five clusters
for cluster_idx in range(5):
    class_shap = shap_array[:, :, cluster_idx]

    # Beeswarm/dot-style SHAP summary
    plt.figure(figsize=(9, 5))
    shap.summary_plot(class_shap, X_train, show=False)
    plt.title(f"Figure {3 + cluster_idx}: Cluster {cluster_idx + 1} SHAP impact")
    plt.tight_layout()
    plt.show()

    # Bar summary
    plt.figure(figsize=(9, 5))
    shap.summary_plot(class_shap, X_train, plot_type="bar", show=False)
    plt.title(f"Cluster {cluster_idx + 1}: mean |SHAP|")
    plt.tight_layout()
    plt.show()

print("Generated SHAP plots for all five clusters.")


In [ ]:
# Figure 8-style global cluster map using Plotly.
# This avoids GeoPandas' removed `naturalearth_lowres` helper.

try:
    import plotly.express as px

    map_df = results[["Country", "cluster_display"]].copy()
    map_df["Country"] = map_df["Country"].str.split("; ")
    map_df = map_df.explode("Country", ignore_index=True)

    country_renames = {
        "Islamic Republic of Iran": "Iran",
        "Korea": "South Korea",
        "United States": "United States of America",
        "Czech Republic": "Czechia",
        "Taiwan Province of China": "Taiwan",
        "West Bank and Gaza": "Palestine"
    }
    map_df["Country"] = map_df["Country"].replace(country_renames)
    map_df["Cluster"] = map_df["cluster_display"].astype(str)

    fig = px.choropleth(
        map_df,
        locations="Country",
        locationmode="country names",
        color="Cluster",
        title="Figure 8 reproduction: Global distribution of stock-exchange clusters"
    )
    fig.show()
    print("Global cluster map generated.")
except Exception as exc:
    # Figure 8 is visualization-only. Core clustering, RF, and SHAP results
    # have already been reproduced above.
    print("Map display skipped by this runtime:", type(exc).__name__, str(exc))

print("\nSUCCESS: notebook completed from top to bottom without an uncaught error.")
